<a href="https://colab.research.google.com/github/wadeea23/Facial-Emotion-Recognition.lab/blob/main/wadeea_lab_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import zipfile
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Input, Activation, Flatten
from keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report ,recall_score


**part 1**

In [ ]:
zip_path = "/content/drive/MyDrive/GA/labNN.zip"
extract_path = "/content/labNN"

In [ ]:
test_path = os.path.join(extract_path, "test")
train_path = os.path.join(extract_path, "train")
emotions = os.listdir(train_path) ## to known every feeling how many pic in thair
for i in emotions:
    print(i, len(os.listdir(os.path.join(train_path, i))))

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(emotions, [
    len(os.listdir(os.path.join(train_path, i)))
    for i in emotions])

plt.title("Number of Images per Emotion")
plt.xlabel("Emotion")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(len(emotions)):
    plt.subplot(2, 4, i + 1)
    img = Image.open(os.path.join(train_path, emotions[i], os.listdir(os.path.join(train_path, emotions[i]))[0]))
    plt.imshow(img,cmap="gray")
    plt.title(emotions[i])

the data is imbalanced has far fewer images than the other,
the image are 48x48 ,samplesare mostly clear



In [ ]:
X_train = []
y_train = []
for label,i in enumerate(emotions):
    for j in os.listdir(os.path.join(train_path, i)):
        img = Image.open(os.path.join(train_path, i, j))
        img = img.resize((48, 48))
        img = np.array(img)
        X_train.append(img)
        y_train.append(label)
X_train=np.array(X_train)
y_train=np.array(y_train)
print(X_train.shape,y_train.shape)

In [ ]:
X_test = []
y_test = []
for label,i in enumerate(emotions):
    for j in os.listdir(os.path.join(test_path, i)):
        img = Image.open(os.path.join(test_path, i, j))
        img = img.resize((48, 48))
        img = np.array(img)
        X_test.append(img)
        y_test.append(label)
X_test=np.array(X_test)
y_test=np.array(y_test)
print(X_test.shape,y_test.shape)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
#normalized to 0,1
X_train=X_train/255
X_val=X_val/255
X_test=X_test/255
print(X_train.min(), X_train.max())

**PART 2**
Neural Network

In [ ]:
model = Sequential()
model.add(Input(shape=(48, 48)))
model.add(Flatten())
model.add(Dense(128))
model.add(Activation('relu'))
model.add(Dense(7))
model.add(Activation('softmax'))


In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['acc'])

In [ ]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))

In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
teat_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", teat_loss)

In [ ]:
confusion_matrix(y_test, np.argmax(model.predict(X_test), axis=1))

In [ ]:
classification_report(y_test, np.argmax(model.predict(X_test), axis=1))

In [ ]:
recall_score(y_test, np.argmax(model.predict(X_test), axis=1), average=None)

**part 3 **

In [ ]:
y_pred = model.predict(X_test)

y_pred = np.argmax(y_pred, axis=1)
y_true =np.array(y_test)

error = np.where(y_pred != y_true)[0]
print(error)

for i in error[:5]:
    plt.figure(figsize=(4, 4))
    plt.imshow(X_test[i], cmap='gray')
    plt.title(f"Predicted: {emotions[y_pred[i]]}, True: {emotions[y_true[i]]}")
    plt.show()

The model made mistakes because some emotions look similar. Some images may also be unclear or have different lighting.

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val), callbacks=[early_stopping])

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

**part 4**

In [ ]:
!pip install streamlit pyngrok

In [ ]:

%%writefile /content/app.py

import streamlit as st
import numpy as np
import json
from PIL import Image
from tensorflow.keras.models import load_model

# Load model and emotion names
model = load_model("/content/emotion_model.keras")

with open("/content/emotion_classes.json", "r") as f:
    classes = json.load(f)

# App title
st.title("Facial Emotion Detector")
st.write("Upload an image or open the camera.")

# Upload image
uploaded_image = st.file_uploader(
    "Upload Image",
    type=["jpg", "jpeg", "png"]
)

# Camera button
open_camera = st.checkbox("Open Camera")

camera_image = None

if open_camera:
    camera_image = st.camera_input("Take a Photo")

# Select image
if uploaded_image is not None:
    image_file = uploaded_image
elif camera_image is not None:
    image_file = camera_image
else:
    image_file = None

# Display and predict
if image_file is not None:

    image = Image.open(image_file)
    st.image(image, caption="Your Image", width=300)

    if st.button("Predict Emotion"):

        # Prepare image
        image = image.convert("L")
        image = image.resize((48, 48))
        image = np.array(image) / 255.0
        image = np.expand_dims(image, axis=0)
        image = np.expand_dims(image, axis=-1)

        # Predict emotion
        prediction = model.predict(image, verbose=0)[0]
        best_index = np.argmax(prediction)

        # Show result
        st.subheader("Predicted Emotion")
        st.success(classes[best_index].capitalize())

        # Show probabilities
        st.subheader("Emotion Probabilities")

        for i in range(len(classes)):
            st.write(classes[i].capitalize())
            st.progress(float(prediction[i]))
            st.write(f"{prediction[i] * 100:.2f}%")


In [ ]:
!streamlit run /content/app.py --server.port 8501 &>/content/logs.txt &
!npx localtunnel --port 8501